In [1]:
# No pip installs needed — all libraries below come pre-installed in Colab
import pandas as pd
import numpy as np
import os
from google.colab import files

In [ ]:
uploaded = files.upload()
# kaushil268: Training.csv, Testing.csv
# itachi9604: dataset.csv, Symptom-severity.csv, symptom_Description.csv, symptom_precaution.csv
# dhivyeshrk: Final_Augmented_dataset_Diseases_and_Symptoms.csv
# nautiyalayush: train.csv, test.csv
# Mendeley: SymbiPredict.csv
print("Uploaded files:", list(uploaded.keys()))

Saving SymbiPredict.csv to SymbiPredict.csv
Saving test.csv to test.csv
Saving train.csv to train.csv
Saving Final_Augmented_dataset_Diseases_and_Symptoms.csv to Final_Augmented_dataset_Diseases_and_Symptoms.csv
Saving symptom_precaution.csv to symptom_precaution.csv
Saving symptom_Description.csv to symptom_Description.csv
Saving dataset.csv to dataset.csv
Saving Symptom-severity.csv to Symptom-severity.csv
Saving Training.csv to Training.csv
Saving Testing.csv to Testing.csv
Uploaded files: ['SymbiPredict.csv', 'test.csv', 'train.csv', 'Final_Augmented_dataset_Diseases_and_Symptoms.csv', 'symptom_precaution.csv', 'symptom_Description.csv', 'dataset.csv', 'Symptom-severity.csv', 'Training.csv', 'Testing.csv']


In [2]:
# ---------- kaushil268 ----------
kaushil_train = pd.read_csv("Training.csv")
kaushil_test  = pd.read_csv("Testing.csv")

# ---------- itachi9604 ----------
itachi_main   = pd.read_csv("dataset.csv")
itachi_sev    = pd.read_csv("Symptom-severity.csv")
itachi_desc   = pd.read_csv("symptom_Description.csv")
itachi_prec   = pd.read_csv("symptom_precaution.csv")

# ---------- dhivyeshrk ----------
dhivyeshrk_df = pd.read_csv("Final_Augmented_dataset_Diseases_and_Symptoms.csv")

# ---------- nautiyalayush ----------
nauti_train   = pd.read_csv("train.csv", encoding="Windows-1252")
nauti_test    = pd.read_csv("test.csv", encoding="Windows-1252")

# ---------- SymbiPredict ----------
# Replace filename below if Mendeley gave it a different name
symbi_df      = pd.read_csv("SymbiPredict.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
datasets = {
    "kaushil_train":  kaushil_train,
    "kaushil_test":   kaushil_test,
    "itachi_main":    itachi_main,
    "itachi_sev":     itachi_sev,
    "itachi_desc":    itachi_desc,
    "itachi_prec":    itachi_prec,
    "dhivyeshrk":     dhivyeshrk_df,
    "nauti_train":    nauti_train,
    "nauti_test":     nauti_test,
    "symbi":          symbi_df,
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"{name}  |  Shape: {df.shape}")
    print(f"Columns: {list(df.columns[:10])} ...")
    df.head(2)


kaushil_train  |  Shape: (4920, 134)
Columns: ['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing', 'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity', 'ulcers_on_tongue'] ...

kaushil_test  |  Shape: (42, 133)
Columns: ['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing', 'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity', 'ulcers_on_tongue'] ...

itachi_main  |  Shape: (4920, 18)
Columns: ['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9'] ...

itachi_sev  |  Shape: (133, 2)
Columns: ['Symptom', 'weight'] ...

itachi_desc  |  Shape: (41, 2)
Columns: ['Disease', 'Description'] ...

itachi_prec  |  Shape: (41, 5)
Columns: ['Disease', 'Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4'] ...

dhivyeshrk  |  Shape: (246945, 378)
Columns: ['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symp

In [4]:
def clean_symptom_name(name):
    """Standardize a symptom string to lowercase_with_underscores."""
    if pd.isna(name):
        return None
    name = str(name).strip()          # remove leading/trailing whitespace
    name = name.lower()               # lowercase everything
    name = name.replace(" ", "_")     # spaces → underscores
    name = name.replace("-", "_")     # hyphens → underscores
    name = name.strip("_")            # remove leading/trailing underscores
    return name if name != "" else None

def clean_disease_name(name):
    """Standardize a disease string to Title Case, no underscores."""
    if pd.isna(name):
        return None
    name = str(name).strip()
    name = name.replace("_", " ")
    name = name.title()               # Title Case
    return name

In [5]:
def preprocess_kaushil(train_df, test_df):
    # Combine train and test
    df = pd.concat([train_df, test_df], ignore_index=True)

    # Drop unnamed/junk columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Rename label column to 'disease'
    df = df.rename(columns={"prognosis": "disease"})

    # Clean disease names
    df["disease"] = df["disease"].apply(clean_disease_name)

    # Clean symptom column names
    symptom_cols = [c for c in df.columns if c != "disease"]
    df = df.rename(columns={c: clean_symptom_name(c) for c in symptom_cols})

    # Fill nulls with 0, ensure binary int
    symptom_cols_clean = [c for c in df.columns if c != "disease"]
    df[symptom_cols_clean] = df[symptom_cols_clean].fillna(0).astype(int)

    # Drop duplicates
    df = df.drop_duplicates()

    print(f"kaushil268 preprocessed → Shape: {df.shape}")
    print(f"Diseases: {df['disease'].nunique()} | Symptoms: {len(symptom_cols_clean)}")
    return df

kaushil_clean = preprocess_kaushil(kaushil_train, kaushil_test)
kaushil_clean.head(3)

kaushil268 preprocessed → Shape: (305, 133)
Diseases: 41 | Symptoms: 132


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,disease
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection


In [6]:
def preprocess_itachi_main(df):
    # Clean disease column
    df = df.copy()
    df["Disease"] = df["Disease"].apply(clean_disease_name)

    # Get all symptom columns
    symptom_cols = [c for c in df.columns if "Symptom" in c]

    # Clean all symptom text values
    for col in symptom_cols:
        df[col] = df[col].apply(clean_symptom_name)

    # Collect all unique symptom names across all symptom columns
    all_symptoms = set()
    for col in symptom_cols:
        all_symptoms.update(df[col].dropna().unique())
    all_symptoms = sorted(all_symptoms)

    # Build binary matrix: for each row, mark which symptoms are present
    binary_rows = []
    for _, row in df.iterrows():
        present = set(row[symptom_cols].dropna().values)
        binary_row = {s: (1 if s in present else 0) for s in all_symptoms}
        binary_row["disease"] = row["Disease"]
        binary_rows.append(binary_row)

    result = pd.DataFrame(binary_rows)

    # Move disease column to end
    cols = [c for c in result.columns if c != "disease"] + ["disease"]
    result = result[cols]

    # Drop duplicates
    result = result.drop_duplicates()

    print(f"itachi9604 (main) preprocessed → Shape: {result.shape}")
    print(f"Diseases: {result['disease'].nunique()} | Symptoms: {len(all_symptoms)}")
    return result

itachi_clean = preprocess_itachi_main(itachi_main)
itachi_clean.head(3)

itachi9604 (main) preprocessed → Shape: (304, 132)
Diseases: 41 | Symptoms: 131


,abdominal_pain,abnormal_menstruation,acidity,acute_liver_failure,altered_sensorium,anxiety,back_pain,belly_pain,blackheads,bladder_discomfort,...,watering_from_eyes,weakness_in_limbs,weakness_of_one_body_side,weight_gain,weight_loss,yellow_crust_ooze,yellow_urine,yellowing_of_eyes,yellowish_skin,disease
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection


In [7]:
# --- Severity Lookup ---
itachi_sev.columns = itachi_sev.columns.str.strip()
itachi_sev["Symptom"] = itachi_sev["Symptom"].apply(clean_symptom_name)
itachi_sev["weight"]  = pd.to_numeric(itachi_sev["weight"], errors="coerce").fillna(0).astype(int)
itachi_sev = itachi_sev.rename(columns={"weight": "severity_weight"})
itachi_sev = itachi_sev.dropna(subset=["Symptom"]).drop_duplicates(subset=["Symptom"])
print("Severity lookup shape:", itachi_sev.shape)

# --- Description Lookup ---
itachi_desc.columns = itachi_desc.columns.str.strip()
itachi_desc["Disease"]     = itachi_desc["Disease"].apply(clean_disease_name)
itachi_desc["Description"] = itachi_desc["Description"].astype(str).str.strip()
itachi_desc = itachi_desc.drop_duplicates(subset=["Disease"])
print("Description lookup shape:", itachi_desc.shape)

# --- Precaution Lookup ---
itachi_prec.columns = itachi_prec.columns.str.strip()
itachi_prec["Disease"] = itachi_prec["Disease"].apply(clean_disease_name)
prec_cols = [c for c in itachi_prec.columns if "Precaution" in c]
for col in prec_cols:
    itachi_prec[col] = itachi_prec[col].astype(str).str.strip()
itachi_prec = itachi_prec.drop_duplicates(subset=["Disease"])
print("Precaution lookup shape:", itachi_prec.shape)

Severity lookup shape: (132, 2)
Description lookup shape: (41, 2)
Precaution lookup shape: (41, 5)


In [8]:
def preprocess_dhivyeshrk(df):
    df = df.copy()

    # Drop unnamed index columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Identify disease label column
    print("Columns sample:", list(df.columns[:5]), "...", list(df.columns[-3:]))

    label_col = None
    for candidate in ["label", "disease", "prognosis", "Disease", "diseases"]:
        if candidate in df.columns:
            label_col = candidate
            break
    if label_col is None:
        raise ValueError("Could not find label column. Check column names above.")

    # Standardize to 'disease'
    df = df.rename(columns={label_col: "disease"})
    df["disease"] = df["disease"].apply(clean_disease_name)

    # Clean symptom column names
    symptom_cols = [c for c in df.columns if c != "disease"]
    df = df.rename(columns={c: clean_symptom_name(c) for c in symptom_cols})

    # Fill nulls, ensure binary int
    symptom_cols_clean = [c for c in df.columns if c != "disease"]
    df[symptom_cols_clean] = df[symptom_cols_clean].fillna(0).astype(int)

    # Drop duplicates
    df = df.drop_duplicates()

    print(f"dhivyeshrk preprocessed → Shape: {df.shape}")
    print(f"Diseases: {df['disease'].nunique()} | Symptoms: {len(symptom_cols_clean)}")
    return df

dhivyeshrk_clean = preprocess_dhivyeshrk(dhivyeshrk_df)
dhivyeshrk_clean.head(3)

Columns sample: ['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms'] ... ['ankle stiffness or tightness', 'ankle weakness', 'neck weakness']
dhivyeshrk preprocessed → Shape: (189647, 378)
Diseases: 773 | Symptoms: 377


,disease,anxiety_and_nervousness,depression,shortness_of_breath,depressive_or_psychotic_symptoms,sharp_chest_pain,dizziness,insomnia,abnormal_involuntary_movements,chest_tightness,...,stuttering_or_stammering,problems_with_orgasm,nose_deformity,lump_over_jaw,sore_in_nose,hip_weakness,back_swelling,ankle_stiffness_or_tightness,ankle_weakness,neck_weakness
0,Panic Disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,Panic Disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Panic Disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
def preprocess_nautiyalayush(train_df, test_df):
    df = pd.concat([train_df, test_df], ignore_index=True)

    # Drop unnamed columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Normalize column names (strip spaces, lowercase)
    df.columns = df.columns.str.strip().str.lower()

    # Find and rename label column
    label_col = None
    for candidate in ["prognosis", "prognoses", "disease", "diseases", "label"]:
        if candidate in df.columns:
            label_col = candidate
            break
    if label_col is None:
        raise ValueError(f"Could not find label column. Columns are: {list(df.columns)}")

    df = df.rename(columns={label_col: "disease"})
    df["disease"] = df["disease"].apply(clean_disease_name)

    # Clean symptom column names
    symptom_cols = [c for c in df.columns if c != "disease"]
    df = df.rename(columns={c: clean_symptom_name(c) for c in symptom_cols})

    # Fill nulls, ensure binary int
    symptom_cols_clean = [c for c in df.columns if c != "disease"]
    df[symptom_cols_clean] = df[symptom_cols_clean].fillna(0).astype(int)

    # Drop duplicates
    df = df.drop_duplicates()

    print(f"nautiyalayush preprocessed → Shape: {df.shape}")
    print(f"Diseases: {df['disease'].nunique()} | Symptoms: {len(symptom_cols_clean)}")
    return df

nauti_clean = preprocess_nautiyalayush(nauti_train, nauti_test)
nauti_clean.head(3)

nautiyalayush preprocessed → Shape: (626, 1327)
Diseases: 391 | Symptoms: 1326


,belly_button_that_sticks_out,bulge_in_the_groin_or_scrotum,delayed_sexual_maturity,delayed_teeth,downward_palpebral_slant_to_eyes,"hairline_with_a_""widow's_peak""",mildly_sunken_chest_(pectus_excavatum),mild_to_moderate_cognitive_problems,mild_to_moderate_short_height,poorly_developed_middle_section_of_the_face,...,tiny_white_spots_inside_the_mouth_(koplik_spots),abnormal_strength_and_direction_of_urine_stream,bed_wetting,bleeding_(hematuria)_at_end_of_urination,visible_narrowing_of_the_urethral_opening_in_boys,swelling_and_redness_of_eyelid_edges,slight_blurring_of_vision_due_to_excess_oil_in_tears,frequent_styes(bumps),pressure_in_the_ear,disease
0,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,Aarskog Syndrome
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Aase Syndrome
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Abdominal Aortic Aneurysm


In [10]:
def preprocess_symbi(df):
    df = df.copy()

    # Drop unnamed columns
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    print("SymbiPredict columns:", list(df.columns[:10]), "...")

    # Find label column
    label_col = None
    for candidate in ["disease", "Disease", "prognosis", "label", "Label", "target"]:
        if candidate in df.columns:
            label_col = candidate
            break
    if label_col is None:
        raise ValueError("Could not find label column. Check printed columns above.")

    df = df.rename(columns={label_col: "disease"})
    df["disease"] = df["disease"].apply(clean_disease_name)

    symptom_cols = [c for c in df.columns if c != "disease"]

    # Check if already binary or needs pivot
    sample_vals = df[symptom_cols[0]].dropna().unique()[:5]
    is_binary = all(v in [0, 1, "0", "1"] for v in sample_vals)

    if is_binary:
        # Already binary — just clean column names
        df = df.rename(columns={c: clean_symptom_name(c) for c in symptom_cols})
        symptom_cols_clean = [c for c in df.columns if c != "disease"]
        df[symptom_cols_clean] = df[symptom_cols_clean].fillna(0).astype(int)
    else:
        # Symptom-list format — pivot to binary
        all_symptoms = set()
        for col in symptom_cols:
            df[col] = df[col].apply(clean_symptom_name)
            all_symptoms.update(df[col].dropna().unique())
        all_symptoms = sorted(all_symptoms)

        binary_rows = []
        for _, row in df.iterrows():
            present = set(row[symptom_cols].dropna().values)
            binary_row = {s: (1 if s in present else 0) for s in all_symptoms}
            binary_row["disease"] = row["disease"]
            binary_rows.append(binary_row)

        df = pd.DataFrame(binary_rows)
        cols = [c for c in df.columns if c != "disease"] + ["disease"]
        df = df[cols]

    df = df.drop_duplicates()

    print(f"SymbiPredict preprocessed → Shape: {df.shape}")
    print(f"Diseases: {df['disease'].nunique()}")
    return df

symbi_clean = preprocess_symbi(symbi_df)
symbi_clean.head(3)

SymbiPredict columns: ['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing', 'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity', 'ulcers_on_tongue'] ...
SymbiPredict preprocessed → Shape: (304, 133)
Diseases: 41


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,disease
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection


In [11]:
def merge_all(dfs: list, names: list):
    print("Merging datasets...")

    # Concatenate — pandas auto-aligns on column names, fills missing with NaN
    master = pd.concat(dfs, ignore_index=True, sort=False)

    # Move disease to last column
    cols = [c for c in master.columns if c != "disease"] + ["disease"]
    master = master[cols]

    # Fill all NaN symptom values with 0
    symptom_cols = [c for c in master.columns if c != "disease"]
    master[symptom_cols] = master[symptom_cols].fillna(0).astype(int)

    # Drop full duplicates
    master = master.drop_duplicates()

    # Drop rows with missing disease label
    master = master.dropna(subset=["disease"])

    print(f"\nMaster Dataset Shape: {master.shape}")
    print(f"Total Unique Diseases: {master['disease'].nunique()}")
    print(f"Total Symptom Features: {len(symptom_cols)}")
    print(f"Total Rows: {len(master)}")
    return master

master_df = merge_all(
    dfs   = [kaushil_clean, itachi_clean, dhivyeshrk_clean, nauti_clean, symbi_clean],
    names = ["kaushil268", "itachi9604", "dhivyeshrk", "nautiyalayush", "SymbiPredict"]
)

master_df.head(3)

Merging datasets...

Master Dataset Shape: (190622, 1757)
Total Unique Diseases: 1117
Total Symptom Features: 1756
Total Rows: 190622


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,tiny_white_spots_inside_the_mouth_(koplik_spots),abnormal_strength_and_direction_of_urine_stream,bed_wetting,bleeding_(hematuria)_at_end_of_urination,visible_narrowing_of_the_urethral_opening_in_boys,swelling_and_redness_of_eyelid_edges,slight_blurring_of_vision_due_to_excess_oil_in_tears,frequent_styes(bumps),pressure_in_the_ear,disease
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal Infection


In [13]:
print("=== FINAL MASTER DATASET VALIDATION ===\n")

# 1. Shape
print(f"Shape: {master_df.shape}")

# 2. Null check
null_count = master_df.isnull().sum().sum()
print(f"Total null values: {null_count}  {'' if null_count == 0 else 'FIX NEEDED'}")

# 3. Duplicate check
dup_count = master_df.duplicated().sum()
print(f"Duplicate rows: {dup_count}  {'' if dup_count == 0 else 'FIX NEEDED'}")

# 4. Binary check on symptom columns (efficient)
symptom_cols = [c for c in master_df.columns if c != "disease"]
non_binary = []
for c in symptom_cols:
    vals = set(master_df[c].unique())
    if not vals.issubset({0, 1}):
        non_binary.append(c)

is_binary = (len(non_binary) == 0)
print(f"All symptom values binary (0/1): {is_binary}  {'' if is_binary else 'FIX NEEDED'}")
if not is_binary:
    print("Non-binary columns:", non_binary[:10])  # show a sample

# 5. Disease label check
print(f"Unique diseases: {master_df['disease'].nunique()}")
print(f"Sample diseases: {list(master_df['disease'].unique()[:10])}")

# 6. Lookup table previews
print("\n--- Severity Lookup ---")
display(itachi_sev.head(3))
print("\n--- Description Lookup ---")
display(itachi_desc.head(3))
print("\n--- Precaution Lookup ---")
display(itachi_prec.head(3))


=== FINAL MASTER DATASET VALIDATION ===

Shape: (190622, 1757)
Total null values: 0  
Duplicate rows: 0  
All symptom values binary (0/1): True  
Unique diseases: 1117
Sample diseases: ['Fungal Infection', 'Allergy', 'Gerd', 'Chronic Cholestasis', 'Drug Reaction', 'Peptic Ulcer Diseae', 'Aids', 'Diabetes', 'Gastroenteritis', 'Bronchial Asthma']

--- Severity Lookup ---


,Symptom,severity_weight
0,itching,1
1,skin_rash,3
2,nodal_skin_eruptions,4



--- Description Lookup ---


,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...



--- Precaution Lookup ---


,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,apply calamine,cover area with bandage,nan,use ice to compress itching


In [12]:
# Master training dataset
master_df.to_csv("master_dataset.csv", index=False)

# Lookup tables (for AI explanation layer)
itachi_sev.to_csv("lookup_severity.csv", index=False)
itachi_desc.to_csv("lookup_description.csv", index=False)
itachi_prec.to_csv("lookup_precaution.csv", index=False)

# Download all to your local machine
files.download("master_dataset.csv")
files.download("lookup_severity.csv")
files.download("lookup_description.csv")
files.download("lookup_precaution.csv")

print("All files saved and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files saved and downloaded!
